In [1]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfTransformer , TfidfVectorizer

from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_csv("../data/cleaned_data.csv")
df.head()

,review,sentiment,clean_review,tokens,label
0,One of the other reviewers has mentioned that ...,positive,one reviewer mentioned watching oz episode you...,"['one', 'reviewer', 'mentioned', 'watching', '...",1
1,A wonderful little production. <br /><br />The...,positive,wonderful little production filming technique ...,"['wonderful', 'little', 'production', 'filming...",1
2,I thought this was a wonderful way to spend ti...,positive,thought wonderful way spend time hot summer we...,"['thought', 'wonderful', 'way', 'spend', 'time...",1
3,Basically there's a family where a little boy ...,negative,basically there family little boy jake think t...,"['basically', 'there', 'family', 'little', 'bo...",0
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,petter matteis love time money visually stunni...,"['petter', 'matteis', 'love', 'time', 'money',...",1


In [3]:
X = df.copy()
y = X["label"].values
X.drop(["label"], axis=1, inplace=True)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Train data:",  X_train.shape, y_train.shape)
print("Test data:",  X_test.shape, y_test.shape)

Train data: (40000, 4) (40000,)
Test data: (10000, 4) (10000,)


In [4]:
# TF-IDF
vectorizer = TfidfVectorizer(min_df=10)

X_train_tfidf = vectorizer.fit_transform(X_train["clean_review"])
X_test_tfidf = vectorizer.transform(X_test["clean_review"])

print('X_train_tfidf shape: ', X_train_tfidf.shape)
print('X_test_tfidf shape: ', X_test_tfidf.shape)


X_train_tfidf shape:  (40000, 21367)
X_test_tfidf shape:  (10000, 21367)


In [5]:
sample = X_train_tfidf[:5]

feature_names = vectorizer.get_feature_names_out()

for i, row in enumerate(sample):
    print(f"\nRow {i}:")
    
    indices = row.nonzero()[1]  
    
    for idx in indices:
        print(feature_names[idx], ":", row[0, idx])


Row 0:
thats : 0.08722238046096971
kept : 0.06391505618208328
asking : 0.07948331703035773
many : 0.10915136347028476
fight : 0.058768056576241526
screaming : 0.07979252340412686
match : 0.07294956561817191
swearing : 0.10048788216785072
general : 0.06482053201355263
mayhem : 0.09584259987870977
permeate : 0.12601373399547816
minute : 0.04433700887002388
comparison : 0.07592601131646792
also : 0.03328922266676816
stand : 0.11875763807629877
think : 0.03475553255638991
onedimensional : 0.09189062726946336
character : 0.029655681810306617
little : 0.03734776965268569
depth : 0.06871019820346459
virtually : 0.08153884545685121
impossible : 0.07029846784331911
care : 0.05454788742965214
happens : 0.05909223917469032
badly : 0.06730034741548488
written : 0.0552635502760124
cypher : 0.11634269043882713
director : 0.0830447044583485
hang : 0.08158320913994975
belief : 0.07021952920797467
topic : 0.08171715288780632
done : 0.09192918375736346
much : 0.03193079517357477
better : 0.037874336543

In [6]:
row = X_train_tfidf[0]

indices = row.nonzero()[1]

sorted_data = sorted(
[(feature_names[i], row[0, i]) for i in indices],
key=lambda x: x[1],
reverse=True
)

sorted_data[:10]

[('must', np.float64(0.1352787418882127)),
 ('permeate', np.float64(0.12601373399547816)),
 ('resolutely', np.float64(0.12497975240153335)),
 ('im', np.float64(0.12378186251977144)),
 ('diverting', np.float64(0.1214733213714654)),
 ('startlingly', np.float64(0.1207189556803729)),
 ('stand', np.float64(0.11875763807629877)),
 ('efficiently', np.float64(0.118673490266634)),
 ('comeuppance', np.float64(0.11805328298014521)),
 ('omnipresent', np.float64(0.11745947563829619))]

In [7]:
import pandas as pd

sample = X_train_tfidf[:10]
feature_names = vectorizer.get_feature_names_out()

rows_list = []

for i in range(sample.shape[0]):
    row = sample[i]
    indices = row.nonzero()[1]

    row_data = {'Row': i}

    for idx in indices:
        row_data[feature_names[idx]] = row[0, idx]

    rows_list.append(row_data)

df_nonzero = pd.DataFrame(rows_list)

df_nonzero.fillna('', inplace=True)

df_nonzero

C:\Users\Workstation\AppData\Local\Temp\ipykernel_17060\434302225.py:21: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_nonzero.fillna('', inplace=True)


,Row,thats,kept,asking,many,fight,screaming,match,swearing,general,...,lost,spent,million,location,impressive,told,nothing,promoted,period,human
0,0,0.087222,0.063915,0.079483,0.109151,0.058768,0.079793,0.07295,0.100488,0.064821,...,,,,,,,,,,
1,1,,,,,,,,,,...,,,,,,,,,,
2,2,,,,,,,,,,...,,,,,,,,,,
3,3,,,,,,,,,,...,,,,,,,,,,
4,4,0.049049,,,0.04092,,,,,,...,,,,,,,,,,
5,5,,,,0.060895,,,,,,...,,,,,,,,,,
6,6,,,,,,,,,,...,,,,,,,,,,
7,7,,,,0.03651,0.058972,,,,,...,,,,,,,,,,
8,8,,,,,,,,,,...,,,,,,,,,,
9,9,,,,0.072635,,,,,,...,0.037235,0.044669,0.046701,0.044011,0.046846,0.039733,0.028023,0.066313,0.043179,0.036138


In [8]:
from scipy.sparse import save_npz
X_test.to_csv("../outputs/X_test_reviews.csv", index=False)
# Save TF-IDF sparse matrices
save_npz("../outputs/X_train_tfidf.npz", X_train_tfidf)

save_npz("../outputs/X_test_tfidf.npz", X_test_tfidf)

# Save labels
np.save("../outputs/y_train.npy", y_train)

np.save("../outputs/y_test.npy", y_test)

In [9]:
import joblib

# Save TF-IDF vectorizer
joblib.dump(
    vectorizer,
    '../outputs/tfidf_vectorizer.pkl'
)

['../outputs/tfidf_vectorizer.pkl']